<a href="https://colab.research.google.com/github/toche7/BasicAIAgentColab/blob/main/BasicHarness5_MyCLI_Plus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent: MyCLi ++

## ส่วนที่ 1: การสนทนาและการเรียกใช้งาน API
ในส่วนนี้เรากำหนดค่าเริ่มต้นในการเชื่อมต่อกับ Groq API (เช่น URL และ Model) และสร้างฟังก์ชัน `chat` เพื่อส่งประวัติการสนทนาพร้อมรายการเครื่องมือ (Tools) ไปยังโมเดล AI และรับผลลัพธ์กลับมา


In [1]:
"""Part 1 - Chat and Runtime execution"""
import os
import sys
import httpx
from google.colab import userdata


URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL = "qwen/qwen3.8-27b"

try:
    api_key = userdata.get('GROQ_API_KEY')
except Exception:
    api_key = os.environ.get('GROQ_API_KEY')

def chat(messages: list[dict], tools: list[dict] | None = None) -> dict:
    """POST the conversation with tools configuration to Groq API."""
    if not api_key:
        raise ValueError("GROQ_API_KEY not found.")

    body = {"model": MODEL, "messages": messages, "max_tokens": 800}
    if tools:
        body["tools"] = tools

    headers = {"Authorization": f"Bearer {api_key}"}
    resp = httpx.post(URL, headers=headers, json=body, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]


## ส่วนที่ 2: Engine Structure.

### 2.1 ส่วนควบคุมระบบและการเข้าถึงเครื่องมือ (Configuration Layer)
เซลล์นี้ทำหน้าที่เป็นจุดศูนย์กลางในการกำหนดค่าความปลอดภัยและการเปิด-ปิดฟังก์ชัน โดยเราสามารถกรองรายชื่อเครื่องมือระบบ (System Tools) ที่อนุญาตให้ทำงาน หรือเลือกเปิด/ปิดการโหลดเครื่องมือแบบพลวัต (Dynamic Tools) จากโฟลเดอร์ภายนอกได้โดยตรง

In [2]:
# -*- coding: utf-8 -*-
# --- CONFIGURATION LAYER ---
# คุณสามารถควบคุมการเปิด/ปิดเครื่องมือแต่ละส่วนได้ที่นี่

CONFIG = {
    # เปิด/ปิดการทำงานของระบบ Dynamic Tools จากโฟลเดอร์ tools/
    "USE_DYNAMIC_TOOLS": True,

    # ระบุรายชื่อของเครื่องมือระบบหลัก (System Tools) ที่อนุญาตให้ทำงานได้
    # (เช่น 'read_file', 'write_file', 'replace_in_file', 'run_terminal_command')
    # หากคุณเว้นว่างไว้ จะถือว่าปิดการใช้งานเครื่องมือระบบหลักทั้งหมด
    "ALLOWED_SYSTEM_TOOLS": [
        "read_file",
        "write_file",
        "replace_in_file",
        "run_terminal_command"
    ]
  #  "ALLOWED_SYSTEM_TOOLS": []

}

### 2.2 เครื่องมือคำนวณคณิตศาสตร์ (Dynamic Tool: calc)
เซลล์นี้จำลองการเขียนโมดูลเครื่องมือแบบ Dynamic ชื่อ `calc.py` เก็บไว้ภายในโฟลเดอร์ `tools/` มีจุดเด่นคือการกำหนดโครงสร้างข้อมูล `SCHEMA` เพื่ออธิบายความสามารถให้ AI ทราบ และรองรับการประมวลผลทางคณิตศาสตร์อย่างปลอดภัย

In [3]:
import os
# สร้างโฟลเดอร์ tools สำหรับเก็บเครื่องมือย่อยแบบโมดูล
os.makedirs('tools', exist_ok=True)
print("Created 'tools' directory successfully!")

Created 'tools' directory successfully!


In [4]:
%%writefile tools/calc.py
# -*- coding: utf-8 -*-
import re

# กำหนด SCHEMA ประจำเครื่องมือ
SCHEMA = {
    "type": "function",
    "function": {
        "name": "calc",
        "description": "Evaluate an arithmetic expression like '12 * (3 + 4)'.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"]
        }
    }
}

# โค้ดการทำงานหลักของเครื่องมือ
def calc(expression: str) -> str:
    """Evaluate an arithmetic expression securely by limiting allowed characters."""
    if not re.match(r'^[0-9+\-*/().\s]+$', expression):
        raise ValueError("only numbers and + - * / are allowed")
    try:
        return str(eval(expression, {"__builtins__": None}, {}))
    except Exception as e:
        raise ValueError(f"Invalid math expression: {e}")


Overwriting tools/calc.py


### 2.3 เครื่องมือเขียนโค้ดเพื่อสร้างเครื่องมือใหม่ (Dynamic Tool: create_tool)
โมดูลหลักตัวสำคัญในโฟลเดอร์ `tools/create_tool.py` ที่ช่วยเพิ่มความยืดหยุ่นให้ระบบ โดยระบบหลักสามารถเรียกเครื่องมือนี้เพื่อเขียนไฟล์ Python ขึ้นมาใหม่ในโฟลเดอร์ `tools/` และทำการอัปโหลดระบบย่อยเข้ามาแบบอัตโนมัติขณะโปรแกรมกำลังทำงาน

In [5]:
%%writefile tools/create_tool.py
# -*- coding: utf-8 -*-
import os
import json
from pathlib import Path

# กำหนด SCHEMA ประจำเครื่องมือเพื่อบอกเอเจนต์หลัก
SCHEMA = {
    "type": "function",
    "function": {
        "name": "create_tool",
        "description": "Automatically create a new dynamic python tool file inside the 'tools' directory with correct structure (including SCHEMA and implementation function) and reloads tools automatically.",
        "parameters": {
            "type": "object",
            "properties": {
                "tool_name": {
                    "type": "string",
                    "description": "The technical filename & function name of the tool, e.g., 'get_weather' (will write to tools/get_weather.py)"
                },
                "tool_description": {
                    "type": "string",
                    "description": "Detailed description of what the tool does for the LLM schema."
                },
                "parameters_properties": {
                    "type": "object",
                    "description": "A JSON object representing properties in JSON schema format (e.g. {'location': {'type': 'string', 'description': 'City name'}})"
                },
                "required_parameters": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of required parameter names."
                },
                "python_code_implementation": {
                    "type": "string",
                    "description": "The Python code representing the body/logic of the function. DO NOT write the 'def tool_name(...)' wrapper, just the inside logic or additional helper imports/functions. The final line should ideally return a string. Note: The input parameters will be passed as local variables of the same name."
                }
            },
            "required": ["tool_name", "tool_description", "parameters_properties", "required_parameters", "python_code_implementation"]
        }
    }
}

def create_tool(tool_name: str, tool_description: str, parameters_properties: dict, required_parameters: list, python_code_implementation: str) -> str:
    """Creates a new dynamic Python tool within the 'tools' directory and validates it."""
    # 1. Clean up and format variables
    tool_name = tool_name.strip().lower().replace(" ", "_").replace(".py", "")
    target_file = Path("./tools") / f"{tool_name}.py"

    # 2. Build template content
    param_str = ", ".join(required_parameters)
    # Formatting string template for dynamic python tool file
    code_content = f"""# -*- coding: utf-8 -*-
import json

SCHEMA = {{
    \"type\": \"function\",
    \"function\": {{
        \"name\": \"{tool_name}\",
        \"description\": \"{tool_description}\",
        \"parameters\": {{
            \"type\": \"object\",
            \"properties\": {json.dumps(parameters_properties, ensure_ascii=False)},
            \"required\": {json.dumps(required_parameters)}
        }}
    }}
}}

def {tool_name}({param_str}) -> str:
    \"\"\"{tool_description}\"\"\"
"""

    # Add implementation indent
    indented_impl = "\n".join(f"    {line}" for line in python_code_implementation.strip().split("\n"))
    code_content += indented_impl + "\n"

    # 3. Write file to disk
    try:
        target_file.parent.mkdir(parents=True, exist_ok=True)
        target_file.write_text(code_content, encoding="utf-8")
    except Exception as e:
        return f"Failed to write file {target_file.name}: {e}"

    # 4. Trigger reload of agent tools
    try:
        import agent_commands
        reload_status = agent_commands.reload_tools()
        return f"Tool '{tool_name}' created successfully at {target_file}! Reload status: {reload_status}"
    except Exception as e:
        return f"Tool created at {target_file} but failed to trigger runtime reload: {e}"


Overwriting tools/create_tool.py


### 2.4 ตัวจัดการโหลดเครื่องมือแบบไดนามิก (Dynamic Loader Module)
นี่คือโมดูลอิสระ (`dynamic_loader.py`) ที่รับหน้าที่สแกนไดเรกทอรี `tools/` และทำการนำเข้าข้อมูล (Import) ของโมดูลอื่นๆ เข้ามาลงทะเบียนในหน่วยความจำอย่างปลอดภัย ช่วยแบ่งแยกหน้าที่ของโค้ดให้สอดคล้องตามหลัก Modular Design

In [6]:
%%writefile dynamic_loader.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Module dedicated to dynamically scanning and importing tools from the 'tools/' directory.
"""
import sys
import importlib.util
from pathlib import Path
from typing import Any, Callable, Dict, List, Tuple

def load_dynamic_tools(tools_dir_path: str = "./tools") -> Tuple[List[Dict[str, Any]], Dict[str, Callable]]:
    """Scans the tools directory and returns imported dynamic schemas and function mappings."""
    dynamic_schemas = []
    dynamic_functions = {}

    tools_dir = Path(tools_dir_path)
    if not (tools_dir.exists() and tools_dir.is_dir()):
        return dynamic_schemas, dynamic_functions

    for filepath in tools_dir.glob("*.py"):
        if filepath.name == "__init__.py":
            continue

        module_name = f"tools.{filepath.stem}"
        try:
            spec = importlib.util.spec_from_file_location(module_name, filepath)
            if spec and spec.loader:
                module = importlib.util.module_from_spec(spec)
                sys.modules[module_name] = module
                spec.loader.exec_module(module)

                # Load the SCHEMA and matching function
                if hasattr(module, "SCHEMA"):
                    schema = getattr(module, "SCHEMA")
                    fn_name = schema["function"]["name"]
                    if hasattr(module, fn_name):
                        dynamic_schemas.append(schema)
                        dynamic_functions[fn_name] = getattr(module, fn_name)
                        print(f"Loaded dynamic tool: {fn_name} from {filepath.name}")
                    else:
                        print(f"Warning: {filepath.name} has SCHEMA but is missing function '{fn_name}'")
        except Exception as e:
            print(f"Failed to load dynamic tool from {filepath.name}: {e}")

    return dynamic_schemas, dynamic_functions

Overwriting dynamic_loader.py


### 2.5 แกนกลางและระบบจัดการเครื่องมือหลัก (System Tools Management)
โมดูลหลักระดับระบบ (`systemtools.py`) ที่ทำการรวมฟังก์ชันเบื้องต้นของเครื่องมือหลัก (เช่น การอ่านเขียนไฟล์ และรัน Terminal Command) เข้ากับเครื่องมือเสริมที่ประมวลผลดึงเข้ามาโดย `dynamic_loader` ตามการตรวจสอบสิทธิ์และอนุญาตผ่านตัวแปร `CONFIG`

In [7]:
%%writefile systemtools.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Part 1 - Tool Definitions (Dynamic Directory Loader version controlled by CONFIG cell)
"""

import json
import subprocess
from pathlib import Path
from typing import Any, Callable, Dict

# --- 1. Base Native Tools (Still available directly) ---
def read_file(path: str) -> str:
    """Read and return the text content of a file."""
    p = Path(path)
    if not p.is_file():
        return f"Error: File not found: {p}"
    return p.read_text(encoding="utf-8")

def write_file(path: str, content: str) -> str:
    """Create or overwrite a file with the given content."""
    p = Path(path)
    if p.parent != Path('.'):
        p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return f"Successfully wrote {len(content)} characters to {p}"

def replace_in_file(path: str, old_text: str, new_text: str) -> str:
    """Replace occurrences of old_text with new_text in a file."""
    p = Path(path)
    if not p.is_file():
        return f"Error: File not found: {p}"
    content = p.read_text(encoding="utf-8")
    if old_text not in content:
        return f"Error: The exact text '{old_text}' was not found in the file."
    content = content.replace(old_text, new_text)
    p.write_text(content, encoding="utf-8")
    return f"Successfully replaced text in {p}"

def run_terminal_command(command: str) -> str:
    """Run a shell command and return stdout + stderr + exit code."""
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
        timeout=120,
    )
    out = []
    out.append(f"Exit code: {result.returncode}")
    if result.stdout:
        out.append(f"Stdout:\n{result.stdout}")
    if result.stderr:
        out.append(f"Stderr:\n{result.stderr}")
    return "\n".join(out) if out else "No output"

# --- Core mapping of all possible system tools ---
_ALL_SYSTEM_SCHEMAS = {
    "read_file": {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a text file in the working directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path of the file to read."}
                },
                "required": ["path"],
            },
        },
    },
    "write_file": {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write or overwrite a text file in the working directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path of the file to write."},
                    "content": {"type": "string", "description": "Content to write."}
                },
                "required": ["path", "content"],
            },
        },
    },
    "replace_in_file": {
        "type": "function",
        "function": {
            "name": "replace_in_file",
            "description": "Replace specific text in a file without overwriting the rest of it.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path of the file to modify."},
                    "old_text": {"type": "string", "description": "The exact text to be replaced."},
                    "new_text": {"type": "string", "description": "The new text to insert in place of old_text."}
                },
                "required": ["path", "old_text", "new_text"],
            },
        },
    },
    "run_terminal_command": {
        "type": "function",
        "function": {
            "name": "run_terminal_command",
            "description": "Execute a bash/Linux command and get output.",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "The command to execute."}
                },
                "required": ["command"],
            },
        },
    },
}

_ALL_SYSTEM_FUNCTIONS = {
    "read_file": read_file,
    "write_file": write_file,
    "replace_in_file": replace_in_file,
    "run_terminal_command": run_terminal_command,
}

# --- Initialize empty lists that will be filled based on CONFIG ---
TOOL_SCHEMAS: list[Dict[str, Any]] = []
FUNCTIONS: Dict[str, Callable] = {}

# 1. Load System Tools using active configuration from IPython environment if available
try:
    from __main__ import CONFIG
except ImportError:
    # Fallback default configuration if CONFIG cell is not initialized
    CONFIG = {
        "USE_DYNAMIC_TOOLS": True,
        "ALLOWED_SYSTEM_TOOLS": ["read_file", "write_file", "replace_in_file", "run_terminal_command"]
    }

allowed_system_list = CONFIG.get("ALLOWED_SYSTEM_TOOLS", [])
for tool_name in allowed_system_list:
    if tool_name in _ALL_SYSTEM_SCHEMAS and tool_name in _ALL_SYSTEM_FUNCTIONS:
        TOOL_SCHEMAS.append(_ALL_SYSTEM_SCHEMAS[tool_name])
        FUNCTIONS[tool_name] = _ALL_SYSTEM_FUNCTIONS[tool_name]

# 2. Dynamic Tool Loader Integration
use_dyn = CONFIG.get("USE_DYNAMIC_TOOLS", True)
if use_dyn:
    try:
        import dynamic_loader
        dyn_schemas, dyn_functions = dynamic_loader.load_dynamic_tools()
        TOOL_SCHEMAS.extend(dyn_schemas)
        FUNCTIONS.update(dyn_functions)
    except Exception as e:
        print(f"Could not import or execute dynamic loader: {e}")

Overwriting systemtools.py



## ส่วนที่ 3: วงจรการทำงานของเอเจนต์ (Agent Loop)
ฟังก์ชัน `run_agent` จะทำหน้าที่ควบคุมขั้นตอนการทำงาน โดยจะถามโมเดล AI หากโมเดลต้องการเรียกใช้เครื่องมือ (Tool Calls) เอเจนต์จะรันเครื่องมือนั้นและส่งผลลัพธ์กลับไปให้โมเดลประมวลผลต่อ ทำซ้ำแบบนี้เรื่อยๆ จนกว่าจะได้คำตอบสุดท้าย


In [8]:
%%writefile agent_commands.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Persistent slash-command layer for the agent.
"""
import importlib
import importlib.util
import json
import sys
from pathlib import Path

_TOOLS_MODULE = "systemtools"
_tools_path = Path(__file__).with_name(_TOOLS_MODULE + ".py")

class AgentTools:
    def __init__(self):
        self.schemas: list = []
        self.functions: dict = {}
        self.source_path = str(_tools_path)

    def names(self):
        return list(self.functions.keys())

AGENT_TOOLS = AgentTools()

def _load_tools_module():
    if _TOOLS_MODULE in sys.modules:
        try:
            return importlib.reload(sys.modules[_TOOLS_MODULE])
        except Exception:
            del sys.modules[_TOOLS_MODULE]

    spec = importlib.util.spec_from_file_location(_TOOLS_MODULE, _tools_path)
    if spec and spec.loader:
        module = importlib.util.module_from_spec(spec)
        sys.modules[_TOOLS_MODULE] = module
        spec.loader.exec_module(module)
        return module
    raise ImportError(f"Cannot load {_TOOLS_MODULE}")

def reload_tools():
    try:
        mod = _load_tools_module()
        AGENT_TOOLS.schemas = list(getattr(mod, "TOOL_SCHEMAS", []))
        AGENT_TOOLS.functions = dict(getattr(mod, "FUNCTIONS", {}))
        return f"Reloaded tools successfully. Available: {AGENT_TOOLS.names()}"
    except Exception as e:
        return f"Failed to reload tools: {e}"

def get_schemas():
    return list(AGENT_TOOLS.schemas)

def dispatch(name: str, arguments_json: str) -> str:
    if name not in AGENT_TOOLS.functions:
        return f"Error: Unknown tool '{name}'. Available: {AGENT_TOOLS.names()}"
    try:
        args = json.loads(arguments_json) if arguments_json else {}
        return str(AGENT_TOOLS.functions[name](**args))
    except Exception as e:
        return f"Error executing {name}: {type(e).__name__}: {e}"

# Initialize on module load
reload_tools()


Overwriting agent_commands.py


In [9]:
"""Step 3 (new file): the agent loop. Call the model, run requested tools, repeat until it answers."""
import agent_commands
import importlib

def run_agent(messages: list[dict], max_steps: int = 20) -> str:
    # importlib.reload(agent_commands) # Ensure latest agent_commands is loaded
    # Intercept slash commands
    last_msg = messages[-1]["content"].strip() if messages and messages[-1]["role"] == "user" else ""
    if last_msg.startswith("/reload"):
        result = agent_commands.reload_tools()
        messages.append({"role": "assistant", "content": result})
        return result

    for _ in range(max_steps):  # max_steps prevents infinite loops
        # Use dynamic schemas from agent_commands
        schemas = agent_commands.get_schemas()
        reply = chat(messages, schemas if schemas else None)

        calls = reply.get("tool_calls") or []
        messages.append({"role": "assistant", "content": reply.get("content") or "",
                         **({"tool_calls": calls} if calls else {})})

        if not calls:
            return messages[-1]["content"]

        for call in calls:
            fn = call["function"]
            print(f"[tool] {fn['name']}({fn['arguments']})")
            # Use dynamic dispatch from agent_commands
            result = agent_commands.dispatch(fn["name"], fn["arguments"])
            messages.append({"role": "tool", "tool_call_id": call["id"],
                             "content": result})

    return f"[stopped after {max_steps} steps without a final answer]"


Loaded dynamic tool: calc from calc.py
Loaded dynamic tool: weather_check from weather_check.py
Loaded dynamic tool: url_info from url_info.py
Loaded dynamic tool: web_search from web_search.py
Failed to load dynamic tool from py_plot.py: unterminated string literal (detected at line 11) (py_plot.py, line 11)
Loaded dynamic tool: py_run from py_run.py
Loaded dynamic tool: create_tool from create_tool.py



## ส่วนที่ 4: คลาสเซสชันการสนทนา (Chat Session Class)
สร้างคลาส `ChatSession` เพื่อเก็บประวัติการคุยทั้งหมด (Conversation History) ของผู้ใช้ เพื่อให้เอเจนต์สามารถจดจำบริบทของการสนทนาก่อนหน้านี้ได้


In [10]:
class ChatSession:
    def __init__(self, max_history: int = 20):
        self.max_history = max_history
        self.system_prompt = {
            "role": "system",
            "content": (
                "You are a helpful AI assistant operating in a Jupyter Notebook environment. "
                "If the user mentions /reload, acknowledge that tools have been dynamically reloaded."
                "If the user want to a create tool, use the create_tool to create."
            )
        }
        # Initialize history with the system prompt
        self.messages = [self.system_prompt]

    def send_message(self, user_message: str) -> str:
        # 1. Append the new user query to the history
        self.messages.append({"role": "user", "content": user_message})

        # 2. Run the agent loop starting with the current history
        response = run_agent(self.messages)

        # 3. Sliding Window: Keep system prompt + latest max_history messages
        if len(self.messages) > self.max_history + 1:
            # Keep the first message (system prompt) and the last N messages
            self.messages = [self.system_prompt] + self.messages[-self.max_history:]

        return response



## ส่วนที่ 5: หน้าต่างแชทจำลอง (Interactive Chat Widget UI)
ส่วนสุดท้ายคือการสร้างหน้าต่างแชทโต้ตอบ (UI) สวยงามโดยใช้ `ipywidgets` เพื่อให้ผู้ใช้สามารถพิมพ์คุยกับเอเจนต์ได้แบบเรียลไทม์ใน Notebook


In [11]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Initialize a fresh chat session
ui_session = ChatSession()

# HTML template for chat messages
def format_message(sender, message):
    bg_color = "#E3F2FD" if sender == "You" else "#FFEBEE" if sender == "System Error" else "#F5F5F5"
    align = "right" if sender == "You" else "left"
    margin = "margin-left: 20%;" if sender == "You" else "margin-right: 20%;"
    text_color = "color: #C62828;" if sender == "System Error" else ""
    return f'''
    <div style="text-align: {align}; margin: 5px 0;">
        <div style="display: inline-block; background-color: {bg_color}; padding: 8px 12px;
                    border-radius: 12px; max-width: 80%; text-align: left; {margin} {text_color}">
            <strong>{sender}:</strong> {message.replace('\n', '<br>')}
        </div>
    </div>
    '''

# UI Elements - Fixed overflow property for correct scrolling
output_area = widgets.Output(layout=widgets.Layout(height='300px', overflow='auto', border='1px solid #ccc', padding='10px'))
text_input = widgets.Text(placeholder='Type your message here...', layout=widgets.Layout(width='80%'))
send_button = widgets.Button(description='Send', button_style='primary', layout=widgets.Layout(width='18%'))

# Action on sending a message
def on_send_click(b):
    user_msg = text_input.value.strip()
    if not user_msg:
        return

    # Clear input immediately for better UX
    text_input.value = ''

    # Append user message to UI
    with output_area:
        display(HTML(format_message("You", user_msg)))

    # Get agent response safely
    try:
        agent_reply = ui_session.send_message(user_msg)
        # Append agent reply to UI
        with output_area:
            display(HTML(format_message("Agent", agent_reply)))
    except Exception as e:
        with output_area:
            error_msg = str(e)
            if "429" in error_msg:
                error_msg = "Rate limit reached (HTTP 429). Please wait a brief moment and try again."
            display(HTML(format_message("System Error", error_msg)))

send_button.on_click(on_send_click)
text_input.on_submit(on_send_click) # Send when pressing Enter

# Display the Layout
print("Interactive Chat Widget:")
display(widgets.VBox([output_area, widgets.HBox([text_input, send_button])]))

Interactive Chat Widget:
